# The classic NLP pipeline — a sentiment classifier

_Capstones_

**Walk the path that took natural-language processing from bag-of-words to attention: build a sentiment classifier out of word embeddings, a bidirectional LSTM, and an attention-pooling head.**

---

This notebook is a practice scaffold for the **The classic NLP pipeline — a sentiment classifier** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, random
torch.manual_seed(1); random.seed(1)

# ---------------------------------------------------------------
# 0. A tiny labeled sentiment dataset (toy, generated, ours).
#    Positive sentences contain positive words; negative contain
#    negative words; both are padded out with neutral filler so the
#    model MUST learn to find the sentiment word, not memorize length.
# ---------------------------------------------------------------
pos_words = ["great","love","excellent","amazing","wonderful","fantastic",
             "brilliant","superb","delightful","gripping"]
neg_words = ["terrible","awful","horrible","boring","dreadful","disappointing",
             "tedious","lousy","painful","forgettable"]
neutral   = ["the","a","this","that","movie","film","story","plot","scene","cast",
             "i","we","it","was","is","really","saw","watched","with","and","of",
             "to","on","very","quite","about","some","one","first","last"]

def make(label, n):
    out = []
    for _ in range(n):
        L = random.randint(7, 12)
        words = [random.choice(neutral) for _ in range(L)]
        pool = pos_words if label == 1 else neg_words
        for _ in range(random.randint(1, 2)):          # inject 1-2 sentiment words
            words[random.randrange(L)] = random.choice(pool)
        out.append((" ".join(words), label))
    return out

train = make(1, 120) + make(0, 120)
test  = make(1, 30)  + make(0, 30)
random.shuffle(train)

# ---------------------------------------------------------------
# 1. TOKENIZE: split on spaces, build a word -> id vocabulary.
# ---------------------------------------------------------------
def tok(s): return s.split()
vocab = {"<pad>": 0}
for s, _ in train + test:
    for w in tok(s):
        if w not in vocab: vocab[w] = len(vocab)
V = len(vocab)
def encode(s): return [vocab[w] for w in tok(s)]

def batchify(data):                       # pad to the longest sentence, build a mask
    seqs = [encode(s) for s, _ in data]; L = max(len(x) for x in seqs)
    X = torch.zeros(len(seqs), L, dtype=torch.long)
    mask = torch.zeros(len(seqs), L)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = torch.tensor(s); mask[i, :len(s)] = 1
    y = torch.tensor([lab for _, lab in data])
    return X, mask, y

Xtr, Mtr, ytr = batchify(train)
Xte, Mte, yte = batchify(test)

# ---------------------------------------------------------------
# 2-5. The model: embed -> BiLSTM -> attention pooling -> classify.
# ---------------------------------------------------------------
EMB, HID = 32, 48
class SentimentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(V, EMB, padding_idx=0)            # step 1: word2vec table
        self.lstm = nn.LSTM(EMB, HID, batch_first=True,
                            bidirectional=True)                   # step 2: BiLSTM encoder
        self.attn = nn.Linear(2 * HID, 1)                         # step 4: token score
        self.cls  = nn.Linear(2 * HID, 2)                         # classifier head
    def forward(self, x, mask):
        h, _   = self.lstm(self.emb(x))                           # (B, L, 2H) context per token
        score  = self.attn(h).squeeze(-1)                         # (B, L)  one score per token
        score  = score.masked_fill(mask == 0, -1e9)               # ignore padding
        alpha  = torch.softmax(score, dim=1)                      # weights over tokens, sum to 1
        ctx    = (alpha.unsqueeze(-1) * h).sum(1)                 # step 3: ONE sentence vector
        return self.cls(ctx), alpha                              # logits + the attention weights

net   = SentimentNet()
opt   = torch.optim.Adam(net.parameters(), lr=4e-3, weight_decay=1e-4)
lossf = nn.CrossEntropyLoss()

# ---------------------------------------------------------------
# Train.
# ---------------------------------------------------------------
for ep in range(80):
    perm = torch.randperm(len(ytr))
    for i in range(0, len(ytr), 32):
        idx = perm[i:i + 32]
        logits, _ = net(Xtr[idx], Mtr[idx])
        loss = lossf(logits, ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()

# ---------------------------------------------------------------
# Result 1: test accuracy.
# ---------------------------------------------------------------
net.eval()
with torch.no_grad():
    acc = (net(Xte, Mte)[0].argmax(1) == yte).float().mean().item()
print("test accuracy:", round(acc, 3))            # our run: 0.917

# ---------------------------------------------------------------
# Result 2: attention weights highlight the sentiment words.
# ---------------------------------------------------------------
sent = "i watched the film and it was really amazing and gripping"
x = torch.tensor([encode(sent)]); m = torch.ones_like(x, dtype=torch.float)
with torch.no_grad():
    logits, alpha = net(x, m)
print("sentence:", sent)
print("prediction:", "positive" if logits.argmax(1).item() == 1 else "negative")
for w, a in zip(tok(sent), alpha[0].tolist()):
    bar = "#" * int(round(a * 50))
    print(f"  {w:10s} {a:.3f} {bar}")
# Our run: the two largest weights land on 'gripping' (0.237) and 'amazing'
# (0.136) -- the model learned to read sentiment WHERE it lives, not the filler.

## Visualize the data & results

_Does the finished pipeline classify sentiment correctly, and do its attention weights actually land on the sentiment-bearing words?_

In [ ]:
# Same build as the CODE cell. After training, we read off the attention
# weights over one labeled sentence and plot them. (PyTorch, seed 1.)
sent = "i watched the film and it was really amazing and gripping"
x = torch.tensor([encode(sent)]); m = torch.ones_like(x, dtype=torch.float)
with torch.no_grad():
    logits, alpha = net(x, m)
print("prediction:", "positive" if logits.argmax(1).item() == 1 else "negative")
print("tokens :", tok(sent))
print("alpha  :", [round(a, 3) for a in alpha[0].tolist()])
# prediction: positive
# tokens : ['i','watched','the','film','and','it','was','really','amazing','and','gripping']
# alpha  : [0.029, 0.035, 0.037, 0.056, 0.065, 0.076, 0.092, 0.091, 0.136, 0.146, 0.237]
# Highest weights -> 'gripping' (0.237) and 'amazing' (0.136). test accuracy: 0.917